In [ ]:
# =============================================================================
# 单元格 1: 初始化、导入模块、参数设定与模型加载
# =============================================================================

import os
import re
import gc
import json
import shutil
import multiprocessing
from itertools import combinations
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- 防止多进程与底层多线程冲突 (超算必备) ---
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import torch
import scipy.spatial.distance

from transformers import AutoTokenizer, AutoModel
from rdkit import Chem, RDLogger
from torch_geometric.data import Data
from IPython.display import display
from tqdm.auto import tqdm # 兼容终端与 Jupyter

import utils

# --- 限制 PyTorch 内部线程数 ---
torch.set_num_threads(1)

# --- 固定随机种子 ---
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"全局随机种子已固定为: {RANDOM_SEED}")

# --- 获取超算可用的 CPU 核心数 ---
# 如果使用 Slurm 调度系统，自动读取分配的核心数
# 获取物理核心数，但强行限制最高并发数为 8 或 16（根据你的节点总内存大小调整）
max_allowed_workers = 368 
detected_cores = int(os.environ.get("SLURM_CPUS_PER_TASK", multiprocessing.cpu_count()))
NUM_WORKERS = min(detected_cores, max_allowed_workers)


# =============================================================================
#                            超参数与文件路径
# =============================================================================

RAW_EMBEDDING_DIM = 768 
TARGET_EMBEDDING_DIM = 768
EMBEDDING_DIM = TARGET_EMBEDDING_DIM 

NUM_COMPLEXES_TO_PROCESS = None
PROCESSING_BATCH_SIZE = 30720 
PROPERTIES_TO_EXCLUDE_FOR_NORM = {'centroid_3d', 'ref_coord_3d', 'Shard ID'}

DATA_ROOT = os.path.join('datas', 'tain_datas', 'SE3TD', 'PDBbind')

pl_index_path = os.path.join(DATA_ROOT, 'index', 'INDEX_general_PL.2025')
nl_index_path = os.path.join(DATA_ROOT, 'index', 'INDEX_general_NL.2025')
pl_data_dir = os.path.join(DATA_ROOT, 'P-L')
nl_data_dir = os.path.join(DATA_ROOT, 'NA-L')

output_dir = os.path.join('processed_data')
PYG_DATA_DIR = os.path.join(output_dir, 'SE3TD_fine_datas_768')
os.makedirs(PYG_DATA_DIR, exist_ok=True) 

VOCAB_FILE = os.path.join(output_dir, 'fragment_vocabulary_768.json')
NORM_STATS_FILE = os.path.join(output_dir, 'normalization_stats_2.9w.json')

print(f"当前工作目录: {os.getcwd()}")
print(f"数据输入根目录: {os.path.abspath(DATA_ROOT)}")
print(f"数据输出目录: {os.path.abspath(output_dir)}")

FEATURE_COLUMNS = [
    'avg_ecc', 'ecc_range',
    'avg_logp', 'avg_tpsa', 'avg_charge', 
    'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable',
    'max_dist_3d', 'max_angle_3d', 'max_plane_angle_3d'
]

# =============================================================================
# 特征归一化策略分类字典
# =============================================================================
ANGLE_FEATURES = {'max_angle_3d': 180.0, 'max_plane_angle_3d': 180.0}
COUNT_FEATURES = {'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable'}

# =============================================================================
#                      ChemBERTa 模型初始化与降维矩阵
# =============================================================================

HF_MODEL_NAME = "seyonec/ChemBERTa-Zinc-Base-v1"
LOCAL_MODEL_PATH = os.path.join(os.getcwd(), "ChemBERTa_Local")

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"[ChemBERTa] 本地未检测到模型，开始从 Hugging Face 下载: {HF_MODEL_NAME} ...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
        model = AutoModel.from_pretrained(HF_MODEL_NAME)
        os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)
        tokenizer.save_pretrained(LOCAL_MODEL_PATH)
        model.save_pretrained(LOCAL_MODEL_PATH)
        print(f"[ChemBERTa] 模型已下载并保存至: {LOCAL_MODEL_PATH}")
    except Exception as e:
        raise RuntimeError(f"模型下载失败，请检查网络连接: {e}")
else:
    print(f"[ChemBERTa] 使用本地模型: {LOCAL_MODEL_PATH}")

print(f"[ChemBERTa] 正在加载模型权重...")
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
model = AutoModel.from_pretrained(LOCAL_MODEL_PATH)
model.eval() 
print(f"[ChemBERTa] 加载完成。")

if TARGET_EMBEDDING_DIM < RAW_EMBEDDING_DIM:
    print(f"[Embedding] 初始化随机投影矩阵: {RAW_EMBEDDING_DIM} -> {TARGET_EMBEDDING_DIM}")
    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    _proj = torch.randn(RAW_EMBEDDING_DIM, TARGET_EMBEDDING_DIM)
    PROJECTION_MATRIX = _proj / torch.norm(_proj, dim=0)
else:
    print(f"[Embedding] 目标维度 >= 原生维度，不进行压缩。")
    PROJECTION_MATRIX = None


# =============================================================================
#                              状态加载
# =============================================================================

try:
    with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)
    vocabulary = set(vocab_data.keys())
    if vocab_data and len(list(vocab_data.values())[0]) != TARGET_EMBEDDING_DIM:
        print(f"警告: 本地词汇表的维度 ({len(list(vocab_data.values())[0])}) 与当前目标维度 ({TARGET_EMBEDDING_DIM}) 不匹配！建议删除旧的 json 文件重新生成。")
    print(f"成功从 '{VOCAB_FILE}' 加载已有词汇表，大小: {len(vocabulary)}")
except (FileNotFoundError, json.JSONDecodeError):
    vocab_data = {} 
    vocabulary = set()
    print(f"未找到或词汇表文件 '{VOCAB_FILE}' 为空，将创建一个新的。")

stats_loaded_from_file = False  
try:
    with open(NORM_STATS_FILE, 'r', encoding='utf-8') as f:
        norm_stats = json.load(f)
    if norm_stats:
        stats_loaded_from_file = True 
    print(f"成功从 '{NORM_STATS_FILE}' 加载已有归一化统计量 (是否跳过重新计算: {stats_loaded_from_file})。")
except (FileNotFoundError, json.JSONDecodeError):
    norm_stats = {}
    print(f"未找到或归一化文件 '{NORM_STATS_FILE}' 为空，将创建一个新的。")

print("-" * 50)
print("初始化全部完成。")


# =============================================================================
# 单元格 2: 辅助函数定义 
# =============================================================================

def get_chemberta_embedding(smiles):
    if not smiles or smiles == "":
        return torch.zeros(TARGET_EMBEDDING_DIM).tolist()
    try:
        inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        if PROJECTION_MATRIX is not None:
            final_embedding = torch.matmul(cls_embedding, PROJECTION_MATRIX)
        else:
            final_embedding = cls_embedding
        return final_embedding.squeeze().tolist()
    except Exception as e:
        return torch.zeros(TARGET_EMBEDDING_DIM).tolist()

def parse_pdbbind_index(file_path, complex_type):
    records = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#'): continue
                parts = re.findall(r'(\S+)', line)
                if len(parts) >= 5:
                    records.append({
                        'pdb_id': parts[0],
                        'release_year': int(parts[2]),
                        'type': complex_type
                    })
    except FileNotFoundError:
        print(f"警告: 索引文件未找到: {file_path}")
    return records

def get_complex_path(row, pl_base_dir, nl_base_dir):
    if row['type'] == 'NL':
        return os.path.join(nl_base_dir, row['pdb_id'])
    elif row['type'] == 'PL':
        year = row['release_year']
        if 1981 <= year <= 2000: 
            year_folder = '1981-2000'
        elif 2001 <= year <= 2010: 
            year_folder = '2001-2010'
        elif 2011 <= year <= 2020: 
            year_folder = '2011-2020'
        else: 
            year_folder = '2021-2024' 
        return os.path.join(pl_base_dir, year_folder, row['pdb_id'])
    return None

def process_single_fragment(fragment, origin_type, pdb_id, id_start_offset):
    try:
        pre_calc_data = utils.prepare_molecule_for_cutting(fragment)
        if not pre_calc_data: return [], "预处理致命错误", 1
        
        main_mol = pre_calc_data['main_mol_for_matching']
        ion_fragments = pre_calc_data['ion_fragments']
        results_storage = []
        dropped_count = 0 
        final_covalent_shards = []

        if main_mol is not None and main_mol.GetNumAtoms() > 0:
            final_covalent_shards = utils.execute_fragmentation_pipeline(main_mol, pre_calc_data['bond_info'])
        else:
            if main_mol is None and not ion_fragments: return [], "主分子重构失败且无离子", 1
            if main_mol is None: dropped_count += 1 

        d_count = utils.calculate_descriptors_and_store(
            storage=results_storage, 
            mol_data={'IDs': pdb_id, 'Smiles': Chem.MolToSmiles(fragment)},
            original_mol_with_conf=fragment, 
            main_mol_for_matching=main_mol if main_mol else fragment,
            final_covalent_fragments=final_covalent_shards, 
            ion_fragments=ion_fragments,
            original_eccentricities=pre_calc_data['original_eccentricities'],
            original_logp_contribs=pre_calc_data['original_logp_contribs'],
            original_tpsa_contribs=pre_calc_data['original_tpsa_contribs'],
            original_charges=pre_calc_data['original_charges']
        )
        dropped_count += d_count

        if results_storage:
            shards_list = []
            processed_result = results_storage[0]
            temp_results = processed_result['results'] + processed_result['ions']
            for i, shard_info in enumerate(temp_results):
                shard_info['origin'] = origin_type
                shard_info['Shard ID'] = id_start_offset + i
                shards_list.append(shard_info)
            return shards_list, None, dropped_count 
            
        return [], "未生成结果", dropped_count
    except Exception as e:
        return [], str(e), 1
    
def save_state_and_report(current_vocab_set, vocab_dict, stats, vocab_file, stats_file, count):
    existing_smiles = set(vocab_dict.keys())
    new_smiles_to_add = current_vocab_set - existing_smiles
    
    if new_smiles_to_add:
        sorted_new_smiles = sorted(list(new_smiles_to_add))
        for smi in sorted_new_smiles:
            vocab_dict[smi] = get_chemberta_embedding(smi)
        tqdm.write(f"  [报告] 新增 {len(sorted_new_smiles)} 个片段到词汇表。")

    if "<UNK>" not in vocab_dict:
        vocab_dict["<UNK>"] = torch.zeros(TARGET_EMBEDDING_DIM).tolist()
    
    temp_dict = vocab_dict.copy()
    unk_embedding = temp_dict.pop("<UNK>")
    sorted_items = sorted(temp_dict.items()) 

    try:
        with open(vocab_file, 'w', encoding='utf-8') as f:
            f.write("{\n")
            f.write(f'  "<UNK>": {json.dumps(unk_embedding)}')
            if sorted_items:
                f.write(",\n")
                for i, (smi, vec) in enumerate(sorted_items):
                    line = f'  {json.dumps(smi, ensure_ascii=False)}: {json.dumps(vec)}'
                    if i < len(sorted_items) - 1:
                        f.write(f"{line},\n")
                    else:
                        f.write(f"{line}\n")
            else:
                f.write("\n")
            f.write("}")
    except Exception as e:
        print(f"保存词汇表时发生错误: {e}")

    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
        
    print(f"--- 进度报告 (已处理 {count} 个复合物): 词汇表({len(sorted_items) + 1}), 统计量已保存 ---")


# =============================================================================
# 新增: 多进程 Worker 核心函数 (运行于独立进程中)
# =============================================================================
def worker_process_complex(complex_data):
    """
    处理单个 PDBbind 复合物。涵盖加载(SDF/Mol2/PDB)、切割和建边。
    处理完毕后自动剥离 RDKit 对象，以支持快速且安全的 Pickle 序列化。
    """
    import scipy.spatial.distance
    lg = RDLogger.logger()
    lg.setLevel(RDLogger.CRITICAL) 
    
    result = {
        'success': False,
        'complex_info': complex_data,
        'processed_groups': None,
        'edges': None,
        'stats': {"success": 0, "rescued_by_mol2": 0, "failed": 0, "dropped_complex": 0}
    }

    # === 阶段 A: 加载3D结构 ===
    sdf_path = complex_data['ligand_sdf_path']
    mol2_path = complex_data['ligand_mol2_path']
    ligand_mol = None
    used_mol2 = False
    
    if os.path.exists(sdf_path):
        try: ligand_mol = Chem.MolFromMolFile(sdf_path, removeHs=True, sanitize=False)
        except: pass
        
    if (ligand_mol is None or ligand_mol.GetNumAtoms() == 0) and os.path.exists(mol2_path):
        try:
            ligand_mol = Chem.MolFromMol2File(mol2_path, removeHs=True, sanitize=False)
            if ligand_mol and ligand_mol.GetNumAtoms() > 0: used_mol2 = True
        except: pass
    
    if not ligand_mol or ligand_mol.GetNumConformers() == 0:
        result['stats']['failed'] += 1; return result
    
    # === [必须补上这行 1]：虽然 SDF/Mol2 质量好，但也可能需要抢救 ===
    ligand_mol = utils.advanced_rescue_valence_errors(ligand_mol)
    
    try:
        ligand_mol.UpdatePropertyCache(strict=False)
        Chem.SanitizeMol(ligand_mol, sanitizeOps=Chem.SANITIZE_SYMMRINGS|Chem.SANITIZE_SETCONJUGATION|Chem.SANITIZE_SETHYBRIDIZATION)
    except:
        result['stats']['failed'] += 1; return result

    pocket_mol = None
    if os.path.exists(complex_data['pocket_path']):
        try: pocket_mol = Chem.MolFromPDBFile(complex_data['pocket_path'], removeHs=True, sanitize=False)
        except: pass
    
    # 顺手把这行优化一下，防止返回空的 Mol 
    if not pocket_mol or pocket_mol.GetNumAtoms() == 0:
        result['stats']['failed'] += 1; return result

    # === [必须补上这行 2]：拯救 PDB 格式的蛋白质口袋！ ===
    pocket_mol = utils.advanced_rescue_valence_errors(pocket_mol)

    raw_pocket_frags = Chem.GetMolFrags(pocket_mol, asMols=True, sanitizeFrags=False)
    pocket_fragments_no_water = []
    for f in raw_pocket_frags:
        is_water = False
        if f.GetNumAtoms() == 1:
            atom = f.GetAtomWithIdx(0)
            if atom.GetAtomicNum() == 8 and atom.GetFormalCharge() == 0: is_water = True
            if f.GetNumHeavyAtoms() > 24: 
                continue
        if not is_water: pocket_fragments_no_water.append(f)
    
    if not pocket_fragments_no_water:
        result['stats']['failed'] += 1; return result

    ligand_fragments = list(Chem.GetMolFrags(ligand_mol, asMols=True, sanitizeFrags=False))
    
    # === 阶段 B: 切割片段与特征计算 ===
    valid_processed_groups = []
    local_shard_id_counter = 0 
    complex_total_dropped_fragments = 0
    
    try:
        all_frags = ligand_fragments + pocket_fragments_no_water
        for original_fragment in all_frags:
            origin_type = 'ligand' if original_fragment in ligand_fragments else 'pocket'
            shards, error, dropped_cnt = process_single_fragment(
                original_fragment, origin_type, 
                complex_data['pdb_id'], local_shard_id_counter
            )
            complex_total_dropped_fragments += dropped_cnt
            
            if shards:
                valid_processed_groups.append({'original_fragment': original_fragment, 'shards': shards})
                local_shard_id_counter += len(shards)
        
        # 严格过滤
        if complex_total_dropped_fragments > 10:
            result['stats']['dropped_complex'] += 1; return result
            
        if valid_processed_groups:
            has_valid_ligand = any(shard['origin'] == 'ligand' for group in valid_processed_groups for shard in group['shards'])
            if not has_valid_ligand:
                result['stats']['dropped_complex'] += 1; return result
        else:
             result['stats']['dropped_complex'] += 1; return result
             
    except Exception:
        result['stats']['dropped_complex'] += 1; return result

    # === 阶段 C: 边构建 ===
    edges = set()
    edge_attributes = {}
    all_shards = [shard for group in valid_processed_groups for shard in group['shards']]
    shard_map = {s['Shard ID']: s for s in all_shards}
    
    for group in valid_processed_groups:
        atom_to_shard_map = {}
        for shard in group['shards']:
            if 'mol' in shard and shard['mol']:
                for atom in shard['mol'].GetAtoms():
                    if atom.GetAtomicNum() != 0 and atom.HasProp("_original_index"):
                        atom_to_shard_map[atom.GetIntProp("_original_index")] = shard['Shard ID']
        main_mol = Chem.RemoveHs(group['original_fragment'])
        for bond in main_mol.GetBonds():
            id1, id2 = atom_to_shard_map.get(bond.GetBeginAtomIdx()), atom_to_shard_map.get(bond.GetEndAtomIdx())
            if id1 is not None and id2 is not None and id1 != id2:
                et = tuple(sorted((id1, id2)))
                if et not in edges:
                    edges.add(et)
                    edge_attributes[et] = 0 if shard_map[id1]['origin'] == 'ligand' else 1
    
    candidate_pairs = []
    for shard1, shard2 in combinations(all_shards, 2):
        if tuple(sorted((shard1['Shard ID'], shard2['Shard ID']))) in edges: continue
        if shard1.get('centroid_3d') and shard2.get('centroid_3d'):
            if scipy.spatial.distance.euclidean(shard1['centroid_3d'], shard2['centroid_3d']) < 12.0:
                candidate_pairs.append((shard1, shard2))
    
    for s1, s2 in candidate_pairs:
        et = tuple(sorted((s1['Shard ID'], s2['Shard ID'])))
        if et in edges: continue
        c1, c2 = utils.get_atom_coordinates(s1.get('clean_mol')), utils.get_atom_coordinates(s2.get('clean_mol'))
        if c1.size > 0 and c2.size > 0 and np.min(scipy.spatial.distance.cdist(c1, c2)) < 6.0:
            edges.add(et)
            origins = sorted([s1['origin'], s2['origin']])
            edge_attributes[et] = 2 if origins == ['ligand', 'pocket'] else (3 if origins == ['ligand', 'ligand'] else 4)

    # ---------------------------------------------------------------------
    # C++ RDKit 对象剥离清理 (极其关键：保护多进程内存安全和传输效率)
    # ---------------------------------------------------------------------
    for group in valid_processed_groups:
        group.pop('original_fragment', None) 
        for shard in group['shards']:
            if 'clean_mol' in shard and shard['clean_mol']:
                try: shard['smiles'] = Chem.MolToSmiles(shard['clean_mol'])
                except: pass
            shard.pop('mol', None)
            shard.pop('clean_mol', None)
            
    result['processed_groups'] = valid_processed_groups
    result['edges'] = edge_attributes
    result['success'] = True
    result['stats']['success'] = 1
    if used_mol2: result['stats']['rescued_by_mol2'] = 1
    return result


# =============================================================================
# 主程序入口 (包含数据准备与多进程调度)
# =============================================================================
if __name__ == '__main__':

    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')

    print("正在解析索引并构建路径...")

    pl_data = parse_pdbbind_index(pl_index_path, 'PL')
    nl_data = parse_pdbbind_index(nl_index_path, 'NL')
    all_complex_data = pl_data + nl_data

    if not all_complex_data:
        raise RuntimeError("未能从索引文件中加载任何复合物数据。请检查路径。")

    df_full = pd.DataFrame(all_complex_data)
    df_full['complex_path'] = df_full.apply(lambda r: get_complex_path(r, pl_data_dir, nl_data_dir), axis=1)

    df_full['ligand_sdf_path'] = df_full.apply(lambda r: os.path.join(r['complex_path'], f"{r['pdb_id']}_ligand.sdf"), axis=1)
    df_full['ligand_mol2_path'] = df_full.apply(lambda r: os.path.join(r['complex_path'], f"{r['pdb_id']}_ligand.mol2"), axis=1)
    df_full['pocket_path'] = df_full.apply(lambda r: os.path.join(r['complex_path'], f"{r['pdb_id']}_pocket.pdb"), axis=1)

    if NUM_COMPLEXES_TO_PROCESS and isinstance(NUM_COMPLEXES_TO_PROCESS, int):
        df_to_process = df_full.head(NUM_COMPLEXES_TO_PROCESS).copy()
    else:
        df_to_process = df_full.copy()
        
    complexes_to_process_list = df_to_process.to_dict('records')

    total_complexes = len(complexes_to_process_list)
    num_batches = (total_complexes + PROCESSING_BATCH_SIZE - 1) // PROCESSING_BATCH_SIZE

    print(f"索引加载完成。")
    print(f"总计 {len(df_full)} 个复合物，本次计划处理 {total_complexes} 个。")
    print(f"分批策略: 每批 {PROCESSING_BATCH_SIZE} 个，共需分为 {num_batches} 个批次执行。")

    final_norm_params = {} 
    total_saved_pyg = 0
    
    # 动态统计逻辑标志位
    needs_calc_stats = not stats_loaded_from_file

    # --- 开始批处理大循环 (主进程协调) ---
    for batch_idx in range(num_batches):
        
        start_idx = batch_idx * PROCESSING_BATCH_SIZE
        end_idx = min((batch_idx + 1) * PROCESSING_BATCH_SIZE, total_complexes)
        current_batch_list = complexes_to_process_list[start_idx : end_idx]
        
        print(f"\n{'='*60}")
        print(f">>> 开始处理批次 {batch_idx + 1} / {num_batches} (使用 {NUM_WORKERS} 进程)")
        print(f"{'='*60}")

        batch_final_data = []
        batch_edge_data = []
        agg_stats = {"success": 0, "rescued_by_mol2": 0, "failed": 0, "dropped_complex": 0}

        # ---------------------------------------------------------------------
        # 阶段 A/B/C: 并行执行重度计算任务 (Pebble 外部强杀与精准追踪版)
        # ---------------------------------------------------------------------
        print(f"  [批次 {batch_idx+1}] 多进程执行阶段 A/B/C (加载/切割/建边)...")
        
        from pebble import ProcessPool
        from concurrent.futures import TimeoutError
        
        # 使用 pebble 的 ProcessPool，它能在 C++ 死锁时直接从 OS 层面杀掉 worker
        with ProcessPool(max_workers=NUM_WORKERS) as pool:
            # 发起映射任务，并设置物理超时为 120 秒
            future = pool.map(worker_process_complex, current_batch_list, timeout=120)
            
            # 获取结果迭代器
            iterator = future.result()
            
            # 手动控制 tqdm 进度条
            pbar = tqdm(total=len(current_batch_list), desc=f"  B{batch_idx+1} 进度")
            
            # [新增] 任务索引计数器，用于精准追踪报错分子
            task_idx = 0 
            
            while True:
                try:
                    # 尝试获取下一个结果
                    res = next(iterator)
                    
                    # --- 正常成功的处理逻辑 ---
                    for k in agg_stats.keys():
                        agg_stats[k] += res['stats'][k]
                        
                    if res['success']:
                        batch_final_data.append({
                            'complex_info': res['complex_info'],
                            'processed_groups': res['processed_groups']
                        })
                        batch_edge_data.append({
                            'complex_info': res['complex_info'],
                            'edges': res['edges']
                        })
                        
                except StopIteration:
                    # 本批次所有任务遍历完成
                    break 
                    
                except TimeoutError:
                    # [拦截成功！] 揪出导致超时的分子信息
                    poison_item = current_batch_list[task_idx]
                    pdb_id = poison_item['pdb_id']
                    
                    agg_stats['dropped_complex'] += 1
                    agg_stats['drop_exception'] += 1
                    
                    # 1. 在终端实时打印警告（不破坏进度条）
                    tqdm.write(f"  [系统击杀] C++ 死锁拦截！分子 PDB: {pdb_id} (耗时 > 120s)")
                    
                    # 2. 写入本地黑名单文件
                    with open("poison_molecules_blacklist.txt", "a") as f:
                        f.write(f"Timeout: PDB={pdb_id}\n")
                        
                except Exception as e:
                    # 捕获其他未知崩溃（比如返回的数据结构不对等）
                    poison_item = current_batch_list[task_idx]
                    pdb_id = poison_item['pdb_id']
                    
                    agg_stats['dropped_complex'] += 1
                    agg_stats['drop_exception'] += 1
                    tqdm.write(f"  [程序崩溃] 分子 PDB: {pdb_id} | 错误信息: {str(e)}")

                finally:
                    # 无论成功、被强杀还是崩溃，进度条推一格，索引加 1，永远保持同步！
                    pbar.update(1) 
                    task_idx += 1
            
            pbar.close()

        print(f"    -> 阶段 A/B/C 统计: 成功 {agg_stats['success']} 个 (其中 Mol2 拯救 {agg_stats['rescued_by_mol2']}), "
              f"加载失败 {agg_stats['failed']}, 质控剔除 {agg_stats['dropped_complex']}")

        # ---------------------------------------------------------------------
        # 阶段 D: 词汇表更新与归一化 (主进程单线程执行)
        # ---------------------------------------------------------------------
        print(f"  [批次 {batch_idx+1}] 阶段 D: 更新词汇表与计算统计量...")
        
        # 遇到首批有效数据时，激活统计更新，并永久关闭后续统计
        if needs_calc_stats and batch_final_data:
            calculate_stats = True
            needs_calc_stats = False 
        else:
            calculate_stats = False
        
        for complex_data in batch_final_data:
            all_shards = [shard for group in complex_data['processed_groups'] for shard in group['shards']]
            for shard in all_shards:
                clean_smi = None
                # 注意：这里我们使用刚才在子进程提取好并存起来的 smiles 字段
                if 'smiles' in shard:
                    clean_smi = shard['smiles']
                
                if clean_smi: vocabulary.add(clean_smi)
                
                if calculate_stats:
                    for prop, value in shard.items():
                        if prop in PROPERTIES_TO_EXCLUDE_FOR_NORM: continue
                        if isinstance(value, (int, float)) and pd.notna(value):
                            st = norm_stats.get(prop, {'count': 0, 'mean': 0.0, 'M2': 0.0})
                            norm_stats[prop] = utils.update_online_stats(st, value)
                            
        save_state_and_report(vocabulary, vocab_data, norm_stats, VOCAB_FILE, NORM_STATS_FILE, end_idx)

        if not final_norm_params and norm_stats:
            print(f"    -> [系统] 正在生成全局归一化参数 (Mean/Std)...")
            for prop, st in norm_stats.items():
                count, mean, M2 = st['count'], st['mean'], st['M2']
                std = np.sqrt(M2 / count) if count >= 2 else 0.0
                final_norm_params[prop] = {'mean': mean, 'std': std}
            print(f"       已准备好 {len(final_norm_params)} 个属性的归一化参数。")
        
        if not final_norm_params and batch_idx > 0:
            print("    -> [警告] 非首批次且无归一化参数，可能是首批次没有有效数据或统计文件为空！")

        # ---------------------------------------------------------------------
        # 阶段 E: PyG 组装与保存
        # ---------------------------------------------------------------------
        print(f"  [批次 {batch_idx+1}] 阶段 E: 生成并保存 PyG 对象...")
        
        batch_saved_count = 0
        data_iterator = zip(batch_final_data, batch_edge_data)
        
        for complex_final_data, complex_edge_data in data_iterator:
            pdb_id = complex_final_data['complex_info']['pdb_id']
            try:
                all_shards = [shard for group in complex_final_data['processed_groups'] for shard in group['shards']]
                all_shards.sort(key=lambda s: s['Shard ID'])

                if not all_shards or any(s.get('centroid_3d') is None or s.get('ref_coord_3d') is None for s in all_shards):
                    continue

                ligand_centroids = np.array([s['centroid_3d'] for s in all_shards if s['origin'] == 'ligand'])
                if len(ligand_centroids) == 0: continue
                ligand_meb_center, _ = utils.calculate_minimum_enclosing_ball(ligand_centroids)
                
                node_features_list, node_pos_list, frag_embeds_list, ref_coords_list = [], [], [], []

                for shard in all_shards:
                    node_type = [1.0, 0.0] if shard['origin'] == 'ligand' else [0.0, 1.0]
                    other_features = []
                    
                    for prop in FEATURE_COLUMNS:
                        value = shard.get(prop, 0.0)
                        
                        if prop in final_norm_params:
                            p = final_norm_params[prop]
                            mean, std = p['mean'], p['std']
                            
                            # === 策略 A: 角度特征 (物理极限映射 [0, 1]) ===
                            if prop in ANGLE_FEATURES:
                                norm_val = float(np.clip(value / ANGLE_FEATURES[prop], 0.0, 1.0))
                                
                            # === 策略 B: 计数特征 (限制最小方差，防止零膨胀爆表) ===
                            elif prop in COUNT_FEATURES:
                                safe_std = max(std, 1.0) # 强制最小 std 为 1.0
                                norm_val = (value - mean) / safe_std
                                norm_val = float(np.clip(norm_val, -6.0, 6.0)) # 保底钳制
                                
                            # === 策略 C: 连续型理化特征 (标准 Z-score + 物理钳制) ===
                            else:
                                safe_std = std if std > 1e-5 else 1.0
                                norm_val = (value - mean) / safe_std
                                norm_val = float(np.clip(norm_val, -6.0, 6.0)) # 强制钳制，扼杀梯度爆炸
                                
                            other_features.append(norm_val)
                        else:
                            other_features.append(value)
                            
                    node_features_list.append(node_type + other_features)
                    node_pos_list.append((np.array(shard['centroid_3d']) - ligand_meb_center).tolist())
                    ref_coords_list.append((np.array(shard['ref_coord_3d']) - ligand_meb_center).tolist())
                    
                    clean_smi = "<UNK>"
                    if 'smiles' in shard: 
                        clean_smi = shard['smiles']
                    
                    if clean_smi not in vocab_data:
                        vocab_data[clean_smi] = get_chemberta_embedding(clean_smi) if clean_smi != "<UNK>" else torch.zeros(TARGET_EMBEDDING_DIM).tolist()
                        vocabulary.add(clean_smi)
                    
                    frag_embeds_list.append(vocab_data[clean_smi])

                x = torch.tensor(node_features_list, dtype=torch.float)
                pos = torch.tensor(node_pos_list, dtype=torch.float)
                frag_embeds = torch.tensor(frag_embeds_list, dtype=torch.float)
                ref_coords = torch.tensor(np.array(ref_coords_list), dtype=torch.float)
                
                edges_dict = complex_edge_data['edges']
                edge_list, edge_attr_list = [], []
                for (id1, id2), et in edges_dict.items():
                    edge_list.append([id1, id2])
                    attr = [0.0]*5; attr[et] = 1.0
                    edge_attr_list.append(attr)
                
                if edge_list:
                    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
                    edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)
                    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
                    edge_attr = torch.cat([edge_attr, edge_attr], dim=0)
                else:
                    edge_index = torch.empty((2, 0), dtype=torch.long)
                    edge_attr = torch.empty((0, 5), dtype=torch.float)

                data = Data(
                    x=x, pos=pos, edge_index=edge_index, edge_attr=edge_attr, 
                    pdb_id=pdb_id, # 统一使用 pdb_id
                    frag_embeds=frag_embeds, ref_coords=ref_coords
                )
                
                # [核心修改 1]：根据 pdb_id 的前两个字符作为子文件夹名
                sub_folder_name = pdb_id[:2] if len(pdb_id) >= 2 else "misc"
                target_dir = os.path.join(PYG_DATA_DIR, sub_folder_name)
                
                # 确保子文件夹存在
                os.makedirs(target_dir, exist_ok=True)
                
                # 将文件存入对应的子文件夹中，以 pdb_id 命名
                torch.save(data, os.path.join(target_dir, f"{pdb_id}.pt"))
                
                batch_saved_count += 1
                total_saved_pyg += 1
                
            except Exception as e:
                continue

        print(f"    -> PyG保存完成: 本批次保存 {batch_saved_count} 个。")
        del batch_final_data
        del batch_edge_data
        gc.collect() 
        
        # [批次收尾]：强制操作系统将内存缓存同步到硬盘 (这部分还在 for 循环内)
        print(f"  [批次 {batch_idx+1}] 正在强制将内存数据落盘，请稍候...")
        os.sync()  # <--- 补回实际执行落盘的代码
        print(f"  [批次 {batch_idx+1}] 完成。内存已清理且已安全落盘。")

        # =========================================================================
        # [全局收尾]：以下代码必须【退格】到最外层（与 for batch_idx 循环平级）
        # =========================================================================
        print(f"\n{'='*60}")
        print(f"�� 全部处理完毕！共成功生成并保存 {total_saved_pyg} 个 PDBbind 复合物的 PyG 图数据。")
        print(f"全局统计结果详见 {NORM_STATS_FILE}，词汇表详见 {VOCAB_FILE}")
        print(f"{'='*60}")